# Five-point screening

GPT-5 template for a relevance score from 1 to 5. Set the run in the configuration cell. `RUN_EXPERIMENT` stays `False` until the first prompt looks right.

The call uses the Responses API with `text.verbosity = low` and `reasoning.effort = minimal`. There is no system message. Every row in `DATA_PATH` is screened. Scores of 3, 4, and 5 count as positive for stage 2. Scores of 4 and 5 count as positive for stage 3.

## Configuration

In [ ]:
import os
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score


In [ ]:
def package_dir():
    cwd = Path.cwd()
    if (cwd / "prompts" / "criteria").exists():
        return cwd
    if (cwd / "Sharing" / "prompts" / "criteria").exists():
        return cwd / "Sharing"
    raise FileNotFoundError("Run this notebook from the Sharing folder, or from the repository root.")

PACKAGE = package_dir()

# Set False to assemble and print the first prompt without calling the API.
RUN_EXPERIMENT = False

MODEL_NAME = "gpt-5"
BATCH_SIZE = 40
DOMAIN = "political_communication"

# Every row of the local file is screened. The files are not in this folder.
# LB: experiments/data/SSOT_manual_LB_20250808_120908.csv
# BM: experiments/data/SSOT_manual_BM_20250813_132621.csv
DATA_PATH = PACKAGE.parent / "experiments" / "data" / "SSOT_manual_LB_20250808_120908.csv"
DATASET_SOURCE = "LB"
TOPIC = "media_diversity"
# For BM set TOPIC = "Computational Text Analysis Methods"

EXPERIMENT_ID = "043"
EXPERIMENT_DATE = "2025-09-15"
USER_PROMPT_ID = "USR_009"
CRITERIA_FILE = PACKAGE / "prompts" / "criteria" / "Criteria_LB_02.csv"
EXAMPLES_FILE = None  # or PACKAGE / "prompts" / "examples" / "exmpl_single_LB_01.csv"

# Other runs with this response format:
# 043 LB 0  Criteria_LB_02.csv  no examples
# 044 LB 1  Criteria_LB_02.csv  exmpl_single_LB_01.csv
# 045 LB 5  Criteria_LB_02.csv  exmpl_five_LB_01.csv
# 052 BM 0  Criteria_BM_02.csv  no examples
# 053 BM 1  Criteria_BM_02.csv  exmpl_single_BM_01.csv
# 054 BM 5  Criteria_BM_02.csv  exmpl_five_BM_01.csv


## Data

In [ ]:
df = pd.read_csv(DATA_PATH)
if "title_full" not in df.columns and "title" in df.columns:
    df["title_full"] = df["title"]

required = ["title_full", "abstract", "stage_2", "stage_3"]
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError("Missing columns: " + ", ".join(missing))

print(f"{DATASET_SOURCE}: {len(df)} rows")
print(f"stage_2 true: {int(df['stage_2'].sum())}")
print(f"stage_3 true: {int(df['stage_3'].sum())}")
df[["title_full", "stage_2", "stage_3"]].head()


## Prompt

In [ ]:
USER_PROMPT_TEMPLATE = '\n### Expert Literature Review Task:\n\nYour task is to evaluate research abstracts on a 5-point relevance scale for a systematic literature review based on provided criteria.\n\n## INSTRUCTIONS:\n1. Carefully read the provided criteria\n2. Review any example abstracts to understand the evaluation pattern, if provided\n3. Apply the criteria to the given abstract and title\n4. Rate relevance on a scale of 1-5:\n   - 5: Highly Relevant - Clearly meets all criteria\n   - 4: Relevant - Meets most criteria\n   - 3: Moderately Relevant - Partial alignment, needs review\n   - 2: Minimally Relevant - Limited alignment\n   - 1: Not Relevant - Does not meet criteria\n5. Provide score and reasoning\n\nBe consistent, objective, and systematic in your evaluation. Do not make up additional criteria beyond what is provided.\n\n## TASK: Screen abstract for {topic} in {domain}\n\n## INCLUSION/EXCLUSION CRITERIA:\n{criteria_text}\n\n{examples_section}\n\n## ABSTRACT TO SCREEN:\n**Title:** {title}\n**Abstract:** {abstract}\n\n## YOUR DECISION:\n\n**Relevance Score:** \n**Reasoning:** '

def load_criteria_text(criteria_file):
    criteria = pd.read_csv(criteria_file)
    text = ""
    inclusion = criteria[criteria["type"] == "inclusion"]
    if len(inclusion):
        text += "**INCLUSION CRITERIA:**\n"
        for _, row in inclusion.iterrows():
            text += f"- **{row['criterion_id']}**: {row['description']}\n"
            if pd.notna(row["examples"]) and str(row["examples"]).strip():
                text += f"  *Examples: {row['examples']}*\n"
    exclusion = criteria[criteria["type"] == "exclusion"]
    if len(exclusion):
        text += "\n**EXCLUSION CRITERIA:**\n"
        for _, row in exclusion.iterrows():
            text += f"- **{row['criterion_id']}**: {row['description']}\n"
            if pd.notna(row["examples"]) and str(row["examples"]).strip():
                text += f"  *Examples: {row['examples']}*\n"
    return text


def load_examples_text(examples_file):
    if not examples_file:
        return ""
    examples = pd.read_csv(examples_file)
    text = "\n## EXAMPLE DECISIONS:\n"
    for _, row in examples.iterrows():
        label = "INCLUDE" if str(row["decision"]).upper() == "INCLUDE" else "EXCLUDE"
        abstract = "" if pd.isna(row["abstract_text"]) else str(row["abstract_text"])
        truncated = False
        if "prompt_was_truncated" in examples.columns and pd.notna(row["prompt_was_truncated"]):
            truncated = str(row["prompt_was_truncated"]).strip().lower() in {"true", "1", "yes"}
        if truncated or len(abstract) > 200:
            shown = abstract[:200] + "..."
        else:
            shown = abstract
        text += f"\n**{label} Example:**\n"
        text += f"*Title:* {row['title']}\n"
        text += f"*Abstract:* {shown}\n"
        text += f"→ **{label}** ({row['reasoning']})\n"
    return text


def build_prompt(title, abstract):
    return USER_PROMPT_TEMPLATE.format(
        topic=TOPIC,
        domain=DOMAIN,
        criteria_text=load_criteria_text(CRITERIA_FILE),
        examples_section=load_examples_text(EXAMPLES_FILE),
        title=title,
        abstract=abstract,
        decision_include="INCLUDE",
        decision_exclude="EXCLUDE",
    )


## API

In [ ]:
load_dotenv(PACKAGE / ".env")
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key) if api_key else None
if client is None:
    print("OPENAI_API_KEY is not set. The prompt preview still works. Copy .env.example to .env before a run.")


def response_text_from(response):
    if response is not None and hasattr(response, "output"):
        for item in response.output:
            content = getattr(item, "content", None)
            if not content:
                continue
            for content_item in content:
                text = getattr(content_item, "text", None)
                if text:
                    return text
    output_text = getattr(response, "output_text", None)
    if output_text:
        return output_text
    return "No response content"

def likert_score_from_text(response_text):
    text = response_text or ""
    for score in ("5", "4", "3", "2", "1"):
        if f"Relevance Score:** {score}" in text or f"**{score}" in text:
            return int(score)
    found = re.findall(r"\b[1-5]\b", text)
    if found:
        return int(found[0])
    return 3


def screen_abstract(prompt):
    if client is None:
        raise RuntimeError("OPENAI_API_KEY is not set.")
    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        text={"verbosity": "low"},
        reasoning={"effort": "minimal"},
    )
    text = response_text_from(response)
    return {"score": likert_score_from_text(text), "reasoning": text}


## Run

In [ ]:
def _scores(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def _print_scores(name, scores):
    print(name)
    print(f"  accuracy {scores['accuracy']:.3f}  precision {scores['precision']:.3f}  recall {scores['recall']:.3f}  f1 {scores['f1']:.3f}")
    print(f"  tp {scores['tp']}  fp {scores['fp']}  tn {scores['tn']}  fn {scores['fn']}")


def _source_id(row, fallback):
    for column in ("ID", "(internal) id", "test_id", "source_id"):
        if column in row.index and pd.notna(row[column]):
            return row[column]
    return fallback


def run_experiment(df, batch_size=BATCH_SIZE):
    items = df.reset_index(drop=True)
    results = []
    batches = (len(items) + batch_size - 1) // batch_size
    print(f"{EXPERIMENT_ID} {DATASET_SOURCE} {USER_PROMPT_ID}: {len(items)} rows, batches of {batch_size}")

    for batch_index in range(batches):
        start = batch_index * batch_size
        stop = min(start + batch_size, len(items))
        print(f"batch {batch_index + 1}/{batches} (rows {start + 1}-{stop})")
        started = time.time()
        for offset, (_, row) in enumerate(items.iloc[start:stop].iterrows()):
            number = start + offset + 1
            record = {
                "example_id": number,
                "source_id": _source_id(row, number),
                "title": row["title_full"],
                "abstract": row["abstract"],
                "stage_2_true": row["stage_2"],
                "stage_3_true": row["stage_3"],
                "stage_2_binary": int(bool(row["stage_2"])),
                "stage_3_binary": int(bool(row["stage_3"])),
                "experiment_id": EXPERIMENT_ID,
                "dataset_source": DATASET_SOURCE,
                "user_prompt_id": USER_PROMPT_ID,
                "model": MODEL_NAME,
                "timestamp": datetime.now().isoformat(),
            }
            try:
                prompt = build_prompt(row["title_full"], row["abstract"])
                result = screen_abstract(prompt)
                score = result["score"]
                record.update({
                    "llm_score": score,
                    "llm_decision": str(score),
                    "llm_reasoning": result["reasoning"],
                    "stage2_llm_binary": int(score >= 3),
                    "stage3_llm_binary": int(score >= 4),
                })
            except Exception as exc:
                print(f"error on row {number}: {exc}")
                record.update({
                    "llm_score": 0,
                    "llm_decision": "ERROR",
                    "llm_reasoning": f"Processing error: {exc}",
                    "stage2_llm_binary": 0,
                    "stage3_llm_binary": 0,
                })

            results.append(record)
        if batch_index < batches - 1:
            print(f"batch finished in {time.time() - started:.1f}s")
            time.sleep(2)

    results_df = pd.DataFrame(results)
    valid = results_df[results_df["llm_decision"] != "ERROR"]
    _print_scores("stage 2, score 3 or higher", _scores(valid["stage_2_binary"], valid["stage2_llm_binary"]))
    _print_scores("stage 3, score 4 or higher", _scores(valid["stage_3_binary"], valid["stage3_llm_binary"]))

    output_dir = PACKAGE / "outputs"
    output_dir.mkdir(exist_ok=True)
    filename = f"{EXPERIMENT_ID}_{DATASET_SOURCE}_{datetime.now().strftime('%m%d%H%M')}.csv"
    path = output_dir / filename
    results_df.to_csv(path, index=False)
    print(f"saved {path}")
    return results_df


In [ ]:
first = df.iloc[0]
print(build_prompt(first["title_full"], first["abstract"]))


In [ ]:
if RUN_EXPERIMENT:
    results = run_experiment(df)
else:
    print("RUN_EXPERIMENT is False. The first prompt is printed above. Set RUN_EXPERIMENT = True to call the API.")
